In [ ]:
"""Verificar GPU y Configuración"""

import torch
import os

# Verificar GPU
print("="*70)
print(" CONFIGURACIÓN DEL ENTORNO")
print("="*70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(" WARNING: No se detectó GPU. El entrenamiento será MUY lento.")

print("="*70)

**Instalar wildlife-tools**

In [ ]:
!pip install -q wildlife-datasets
!pip install -q git+https://github.com/WildlifeDatasets/wildlife-tools

**Explorar dataset**

In [ ]:
from wildlife_datasets import analysis, datasets

datasets.DogFaceNet.get_data('data/DogFaceNet')
dataset = datasets.DogFaceNet('data/DogFaceNet')

dataset.df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import numpy as np

df = dataset.df

id_counts = df['identity'].value_counts()

plt.figure(figsize=(14, 5))

# Histograma de imágenes por perro
plt.subplot(1, 2, 1)
sns.histplot(id_counts.values, bins=30, kde=True, color='skyblue')
plt.title(f'Distribución de imágenes (Total: {len(df)} fotos, {len(id_counts)} perros)')
plt.xlabel('Cantidad de imágenes')
plt.ylabel('Frecuencia')

# Top 10 perros con más fotos
plt.subplot(1, 2, 2)
id_counts.head(10).plot(kind='bar', color='salmon')
plt.title('Top 10 Individuos con más data')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Seleccionar 10 imágenes aleatorias
samples = df.sample(10)

plt.figure(figsize=(15, 6))
for i, (idx, row) in enumerate(samples.iterrows()):
    plt.subplot(2, 5, i+1)
    img_path = os.path.join(dataset.root, row['path'])

    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img)
    else:
        plt.text(0.5, 0.5, "Error loading", ha='center')

    plt.title(f"ID: {row['identity']}")
    plt.axis('off')

plt.suptitle('Muestra Aleatoria del Dataset DogFaceNet')
plt.show()

**Librerías**

In [ ]:
import torch
import torch.nn as nn
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from torchvision.models import resnet50
import torchvision.transforms as T
from torch.optim import SGD
from wildlife_datasets import datasets
from wildlife_tools.data import ImageDataset
from wildlife_tools.train import ArcFaceLoss, BasicTrainer
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.inference import KnnClassifier, TopkClassifier

**Configuraciones**

In [ ]:
# Configuración de rutas
class Config:
    PRETRAINED_BACKBONE = 'ruta_al_backbone'
    OUTPUT_DIR = 'ruta_carpeta_salida'
    DATASET_PATH = 'data/DogFaceNet'

    # Hiperparámetros
    EMBEDDING_DIM = 512
    FREEZE_EPOCHS = 5
    TOTAL_EPOCHS = 100
    BATCH_SIZE = 128
    LR_FROZEN = 0.01      # Mayor LR cuando backbone está congelado
    LR_UNFROZEN = 0.001   # Menor LR cuando se entrena todo

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

**Dividir dataset**

In [ ]:
# Cargar y explorar dataset

datasets.DogFaceNet.get_data(config.DATASET_PATH)
dataset = datasets.DogFaceNet(config.DATASET_PATH)

print(f"   - Total imágenes: {len(dataset.df)}")
print(f"   - Total identidades: {dataset.df['identity'].nunique()}")

# Análisis estadístico
df = dataset.df
id_counts = df['identity'].value_counts()

In [ ]:
# Dividir dataset (Train/Test split)

# Obtener identidades únicas
ids = df['identity'].unique()
print(f"Total IDs: {len(ids)}")

# Shuffle reproducible
rng = np.random.default_rng(seed=42)
rng.shuffle(ids)

# Split 80-20
split = int(0.8 * len(ids))
train_ids = ids[:split]
test_ids = ids[split:]

# Filtrar dataframes
train_df = df[df['identity'].isin(train_ids)].reset_index(drop=True)
test_df = df[df['identity'].isin(test_ids)].reset_index(drop=True)

print(f"Train images: {len(train_df)} ({len(train_ids)} IDs)")
print(f"Test images: {len(test_df)} ({len(test_ids)} IDs)")

# Transformaciones
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# Crear datasets
train_dataset = ImageDataset(train_df, dataset.root, transform=transform)
test_dataset = ImageDataset(test_df, dataset.root, transform=transform)

**Query y Gallery**

In [ ]:
# Query y Gallery

query_rows = []
gallery_rows = []

for identity in test_ids:
    # Obtener todas las imágenes de esta identidad
    id_images = test_df[test_df['identity'] == identity]

    # Si hay más de 1 imagen, dividir
    if len(id_images) > 1:
        # Shuffle las imágenes de esta identidad
        id_images = id_images.sample(frac=1, random_state=42).reset_index(drop=True)

        # Primera mitad para query, segunda mitad para gallery
        mid = len(id_images) // 2
        query_rows.append(id_images.iloc[:mid])
        gallery_rows.append(id_images.iloc[mid:])
    else:
        # Si solo hay 1 imagen, va a gallery
        gallery_rows.append(id_images)

# Concatenar todos los dataframes
query_df = pd.concat(query_rows, ignore_index=True)
gallery_df = pd.concat(gallery_rows, ignore_index=True)

print(f"   Query: {len(query_df)} imágenes de {query_df['identity'].nunique()} IDs")
print(f"   Gallery: {len(gallery_df)} imágenes de {gallery_df['identity'].nunique()} IDs")
print(f"   Overlap de IDs: {len(set(query_df['identity']).intersection(set(gallery_df['identity'])))} IDs")

# Crear datasets
query_dataset = ImageDataset(query_df, dataset.root, transform)
gallery_dataset = ImageDataset(gallery_df, dataset.root, transform)

**Entrenar**

In [ ]:
# Definir arquitectura del modelo

def load_dino_resnet50(weight_path):
    """Carga el backbone DINO pre-entrenado"""
    model = resnet50(weights=None)
    model.fc = nn.Identity()

    state = torch.load(weight_path, map_location="cpu")
    model.load_state_dict(state, strict=False)

    return model

class DinoReID(nn.Module):
    """Modelo completo: DINO backbone + projection head"""

    def __init__(self, backbone, feat_dim=512):
        super().__init__()
        self.backbone = backbone
        self.embed = nn.Linear(2048, feat_dim)
        self.bn = nn.BatchNorm1d(feat_dim)

    def forward(self, x):
        x = self.backbone(x)  # [B, 2048]
        x = self.embed(x)      # [B, 512]
        x = self.bn(x)         # [B, 512]
        return x

In [ ]:
# Entrenar modelo

# Cargar backbone pre-entrenado
backbone = load_dino_resnet50(config.PRETRAINED_BACKBONE)

# Crear modelo completo
model = DinoReID(backbone, feat_dim=config.EMBEDDING_DIM).cuda()

print(f"   - Backbone: ResNet50 (DINO pre-entrenado)")
print(f"   - Embedding dim: {config.EMBEDDING_DIM}")

# Entrenar solo el head (backbone congelado)

# Congelar backbone
for p in model.backbone.parameters():
    p.requires_grad = False

# Verificar que solo head es entrenable
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"   - Parámetros entrenables: {trainable_params:,} / {total_params:,}")

num_train_ids = train_df['identity'].nunique()

objective_phase1 = ArcFaceLoss(
    num_classes=num_train_ids,
    embedding_size=config.EMBEDDING_DIM,
    margin=0.5,
    scale=64
).cuda()

# Optimizador (solo head + ArcFace)
optimizer_phase1 = SGD(
    params=itertools.chain(
        model.embed.parameters(),
        model.bn.parameters(),
        objective_phase1.parameters()
    ),
    lr=config.LR_FROZEN,
    momentum=0.9,
    weight_decay=5e-4
)

trainer_phase1 = BasicTrainer(
    dataset=train_dataset,
    model=model,
    objective=objective_phase1,
    optimizer=optimizer_phase1,
    epochs=config.FREEZE_EPOCHS,
    batch_size=config.BATCH_SIZE,
    num_workers=4,
    device='cuda'
)

trainer_phase1.train()

# Fine-tuning completo (descongelar backbone)

# Descongelar backbone
for p in model.backbone.parameters():
    p.requires_grad = True

# Verificar parámetros entrenables
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   - Parámetros entrenables: {trainable_params:,} / {total_params:,}")

# Reutilizar pesos
objective_phase2 = objective_phase1  # Mantener los pesos aprendidos

# Nuevo optimizador con LR menor
optimizer_phase2 = SGD(
    params=itertools.chain(
        model.parameters(),
        objective_phase2.parameters()
    ),
    lr=config.LR_UNFROZEN,  # LR más bajo
    momentum=0.9,
    weight_decay=5e-4
)

# Trainer
trainer_phase2 = BasicTrainer(
    dataset=train_dataset,
    model=model,
    objective=objective_phase2,
    optimizer=optimizer_phase2,
    epochs=config.TOTAL_EPOCHS - config.FREEZE_EPOCHS,
    batch_size=config.BATCH_SIZE,
    num_workers=4,
    device='cuda'
)

trainer_phase2.train()

# Guardar modelo

final_model_path = os.path.join(config.OUTPUT_DIR, "DINO_supervisado_full.pth")
torch.save(model.state_dict(), final_model_path)

print(f"\n Modelo guardado: {final_model_path}")

**Evaluar**

In [ ]:
# Evaluar

model.eval()

In [ ]:
# Extractor de features
extractor = DeepFeatures(
    model,
    batch_size=32,
    device='cuda'
)

# Extraer embeddings
query_features = extractor(query_dataset)
gallery_features = extractor(gallery_dataset)

# Calcular similitudes
query_labels = np.array(query_dataset.labels)
gallery_labels = np.array(gallery_dataset.labels)

similarity_function = CosineSimilarity()
similarity = similarity_function(query_features, gallery_features)

In [ ]:
# Calcular mAP

def calculate_map(similarity_matrix, query_labels, gallery_labels):
    num_queries = len(query_labels)
    APs = []

    for i in range(num_queries):
        # Obtener similitudes de esta query con toda la gallery
        query_sims = similarity_matrix[i]
        true_label = query_labels[i]

        # Ordenar gallery por similitud (mayor a menor)
        sorted_indices = np.argsort(-query_sims)
        sorted_labels = gallery_labels[sorted_indices]

        # Encontrar cuáles son positivos (misma identidad)
        positives = (sorted_labels == true_label)
        num_positives = positives.sum()

        if num_positives == 0:
            continue

        # Calcular precision en cada posición donde hay un positivo
        precisions = []
        num_correct = 0

        for rank, is_positive in enumerate(positives, start=1):
            if is_positive:
                num_correct += 1
                precision_at_k = num_correct / rank
                precisions.append(precision_at_k)

        AP = np.mean(precisions) if precisions else 0.0
        APs.append(AP)

    # Mean Average Precision
    mAP = np.mean(APs) if APs else 0.0

    return mAP, APs

In [ ]:
# Top-1 y Top-5 accuracy

topk = TopkClassifier(
    database_labels=gallery_labels,
    k=5,
    return_all=False
)

preds = topk(similarity)

# Top-1
top1_correct = preds[:, 0] == query_labels
top1_acc = np.mean(top1_correct)

# Top-5
top5_correct = [query_labels[i] in preds[i] for i in range(len(query_labels))]
top5_acc = np.mean(top5_correct)

# Mean Average Precision
mAP, individual_APs = calculate_map(similarity, query_labels, gallery_labels)

print(f"Top-1 Accuracy: {top1_acc:.4f} ({top1_acc*100:.2f}%)")
print(f"Top-5 Accuracy: {top5_acc:.4f} ({top5_acc*100:.2f}%)")
print(f"Mean Average Precision (mAP): {mAP:.4f} ({mAP*100:.2f}%)")

In [ ]:
# Análisis de errores

# Ver distribución de similitudes
correct_sims = []
incorrect_sims = []

for i in range(len(query_labels)):
    true_label = query_labels[i]
    pred_label = preds[i, 0]

    # Encontrar índice del correcto en gallery
    correct_idx = np.where(gallery_labels == true_label)[0]

    if len(correct_idx) > 0:
        # Similitud con el correcto
        max_correct_sim = similarity[i, correct_idx].max()

        if pred_label == true_label:
            correct_sims.append(max_correct_sim)
        else:
            incorrect_sims.append(max_correct_sim)

if correct_sims and incorrect_sims:
    print(f"   - Similitud promedio (aciertos): {np.mean(correct_sims):.4f}")
    print(f"   - Similitud promedio (errores): {np.mean(incorrect_sims):.4f}")
    print(f"   - Separación: {np.mean(correct_sims) - np.mean(incorrect_sims):.4f}")

In [ ]:
# Guardar solo el backbone para reutilización

backbone_only_path = os.path.join(config.OUTPUT_DIR, "DINO_supervisado_backbone.pth")
torch.save(model.backbone.state_dict(), backbone_only_path)

print(f"\n Backbone guardado: {backbone_only_path}")